# Bronze — ingestão da landing
Incorpora os arquivos Parquet novos da landing com `COPY INTO`. O `COPY INTO` registra os arquivos já carregados em cada tabela de destino, então reexecuções não duplicam dados e cada ambiente controla sua própria carga.

In [ ]:
for nome, padrao in [("catalogo_bronze", "dev_bronze"), ("landing_path", "/Volumes/landing/protheus/arquivos"),
                     ("src_path", ""), ("tabelas", "sc5,sc6,sa1,sa3,sb1")]:
    dbutils.widgets.text(nome, padrao)

import sys

src_path = dbutils.widgets.get("src_path")
if src_path and src_path not in sys.path:
    sys.path.append(src_path)

In [ ]:
from pdc_lib.util import nome_tabela, validar_identificador, validar_tabela_protheus

catalogo_bronze = validar_identificador(dbutils.widgets.get("catalogo_bronze"))
landing_path = dbutils.widgets.get("landing_path").rstrip("/")
if not landing_path.startswith("/Volumes/"):
    raise ValueError("landing_path deve ser um volume do Unity Catalog")
tabelas = [validar_tabela_protheus(t.strip()) for t in dbutils.widgets.get("tabelas").split(",") if t.strip()]

In [ ]:
def diretorio_existe(caminho: str) -> bool:
    try:
        dbutils.fs.ls(caminho)
        return True
    except Exception:
        return False


for tabela in tabelas:
    destino = nome_tabela(catalogo_bronze, "protheus", tabela)
    origem = f"{landing_path}/{tabela}"
    if not diretorio_existe(origem):
        print(f"{tabela}: nenhum arquivo na landing ainda; ignorada.")
        continue
    spark.sql(f"CREATE TABLE IF NOT EXISTS {destino} COMMENT 'Bronze: espelho da tabela {tabela} do Protheus'")
    resultado = spark.sql(f"""
        COPY INTO {destino}
        FROM (
            SELECT *,
                   regexp_extract(_metadata.file_path, 'empresa=([0-9]{{2}})', 1) AS _empresa,
                   _metadata.file_path AS _arquivo,
                   current_timestamp() AS _dt_ingestao
            FROM '{origem}'
        )
        FILEFORMAT = PARQUET
        PATTERN = 'empresa=*/*.parquet'
        FORMAT_OPTIONS ('mergeSchema' = 'true')
        COPY_OPTIONS ('mergeSchema' = 'true')
    """).first()
    print(f"{tabela}: {resultado}")